# B02 — 단일세포 추적과 형태의 시간 변화

**BioCulture | 실제 Dunlop Lab CTC 데이터 | 실습 Notebook v1**

**연구 질문:** 같은 *E. coli* 세포가 시간에 따라 어떻게 변하며, 추적 정답의 부모–자식 연결은 무엇을 뜻하는가?

**학습 목표**: (1) TRA 레이블과 SEG 레이블 구분 (2) `man_track.txt` 검증 (3) TRA ID를 이용한 세포별 면적·길이 시계열 (4) 부모–자식 관계 표시 (5) 추적 결손과 관측 범위의 제약 이해.

실제 데이터: [Dunlop Lab CTC, Zenodo](https://zenodo.org/records/11237127). `CTC.zip` 파일이 `data/bioculture/`에 있어야 합니다. 시간 간격은 제공된 README 기준 **5분/프레임**입니다. **B02에서는 성장률·분열 시간의 생물학적 결론을 확정하지 않습니다.**

**Clean v2 주석 보강판:** 원래의 계산·그림·저장 코드는 변경하지 않고, 교육용 처리 단계 주석만 추가하였다. 실행 완료 v1의 출력값은 참고용으로 유지하며, 새 환경에서는 전체 셀 재실행으로 확인한다.

## 1. 환경·경로 검사
실행 커널이 `Python (BioImage AI)`인지 확인합니다. ZIP을 풀지 않고 직접 읽습니다. `imagecodecs`는 LZW TIFF 복원에 필요합니다.

In [ ]:
# [실행 위치 확인] Jupyter의 현재 작업 폴더를 확인합니다. 파일 경로 오류를 진단할 때 사용합니다.
import os
# os.getcwd()
from pathlib import Path
Path.cwd()

In [ ]:
# [B02-01] 실행 환경과 데이터·출력 경로 설정
# [의존성] ZIP 읽기, 이미지 처리, 데이터 분석에 필요한 모듈을 가져옵니다.
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
try:
    import imagecodecs
    codecs_version = imagecodecs.__version__
except ImportError:
    codecs_version = "not installed (LZW TIFF files may fail)"
from skimage.color import rgb2gray
from skimage.measure import regionprops_table

# [실행 커널 검사] imagecodecs는 LZW 압축 TIFF를 읽을 때 필요한 환경 의존성입니다.
print("Python:", sys.executable)
print("tifffile:", tifffile.__version__, "imagecodecs:", codecs_version)

# [입력 데이터 찾기] Jupyter가 프로젝트 루트 또는 notebooks에서 시작된 상황을 지원합니다.
candidates = [Path.cwd()/"data"/"bioculture"/"CTC.zip", Path.cwd().parent/"data"/"bioculture"/"CTC.zip"]
ZIP_PATH = next((q.resolve() for q in candidates if q.is_file()), None)
if ZIP_PATH is None:
    raise FileNotFoundError("프로젝트 data/bioculture/CTC.zip이 없습니다.")
# [출력 경로] 결과물은 bioimage-ai/results/bioculture/B02 아래에 저장합니다.
PROJECT_ROOT = ZIP_PATH.parent.parent.parent
RESULTS_DIR = PROJECT_ROOT/"results"/"bioculture"/"B02"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# [분석 설정] train/69 시퀀스의 5분 간격 촬영 영상을 이용합니다.
SPLIT, SEQUENCE = "train", "69"
MINUTES_PER_FRAME = 5
# 처음에는 20프레임을 검사합니다. 전체 검증은 None으로 바꿉니다.
MAX_FRAMES = 20
print("ZIP:", ZIP_PATH, "| Output:", RESULTS_DIR)

## 2. 영상과 정답의 프레임별 대응
`SEG/man_segNNN.tif`의 값은 프레임별 **분할 객체 라벨**, `TRA/man_trackNNN.tif`의 값은 **시간 전체에서 유지되는 추적 ID**입니다. 두 값을 동일하다고 가정하지 않습니다.

In [ ]:
# [B02-02] 파일명에서 시점을 파싱하고 영상·SEG·TRA를 매칭
# [CTC 경로 규칙] 영상, SEG 및 TRA 정답은 서로 다른 폴더에 있습니다.
prefix = f"CTC/{SPLIT}/{SEQUENCE}/"
seg_prefix = f"CTC/{SPLIT}/{SEQUENCE}_GT/SEG/"
tra_prefix = f"CTC/{SPLIT}/{SEQUENCE}_GT/TRA/"
track_file = tra_prefix + "man_track.txt"

# [파일 매칭] 파일명에 들어 있는 정수 프레임 번호를 사전 키로 사용합니다. 중복을 오류로 처리합니다.
def by_frame(names, folder, stem):
    pattern = re.compile(r"/" + re.escape(stem) + r"(\d+)\.tif$", re.I)
    out = {}
    for name in names:
        if not name.startswith(folder):
            continue
        m = pattern.search(name)
        if m:
            t = int(m.group(1))
            if t in out:
                raise ValueError(f"중복 frame {t}: {name}")
            out[t] = name
    return out

# [ZIP 목차 읽기] 대용량 ZIP을 압축 해제하지 않고 내부 경로만 탐색합니다.
with ZipFile(ZIP_PATH) as z:
    all_names = z.namelist()
image_files = by_frame(all_names, prefix, "t")
seg_files = by_frame(all_names, seg_prefix, "man_seg")
tra_files = by_frame(all_names, tra_prefix, "man_track")
assert track_file in all_names, "TRA/man_track.txt가 없습니다."
# [세 종류의 프레임 교집합] 영상·SEG·TRA가 모두 존재하는 시점만 분석합니다.
frames = sorted(set(image_files) & set(seg_files) & set(tra_files))
assert frames, "영상, SEG, TRA가 함께 있는 프레임이 없습니다."
print("Image/SEG/TRA:", len(image_files), len(seg_files), len(tra_files))
print("Three-way paired frames:", len(frames))
print("Missing SEG:", sorted(set(image_files)-set(seg_files))[:15])
print("Missing TRA:", sorted(set(image_files)-set(tra_files))[:15])
# [실습 범위] 기본은 처음 20개 프레임이며 MAX_FRAMES=None이면 전부 분석합니다.
selected_frames = frames if MAX_FRAMES is None else frames[:MAX_FRAMES]
print("Frames selected:", len(selected_frames), "from", selected_frames[0], "to", selected_frames[-1])

# [압축 TIFF 처리] ZIP 내부 바이트를 메모리에서 TIFF로 읽습니다.
def read_tif(z, name):
    return tifffile.imread(BytesIO(z.read(name)))

# [차원 통일] RGB/회색조 원본을 2차원 강도 영상으로 변환하되, SEG/TRA 라벨은 변경하지 않습니다.
def gray_image(img):
    if img.ndim == 2: return img
    if img.ndim == 3 and img.shape[-1] in (3,4): return rgb2gray(img[..., :3])
    raise ValueError(f"지원하지 않는 영상 형태: {img.shape}")

## 3. 추적 계보표의 문법과 관계 검증
`man_track.txt`는 `cell_id / start / end / parent_id` 네 열입니다. `parent_id=0`은 부모 정보 없음이지, 생물학적으로 독립 기원이라는 뜻이 아닙니다. 파일에 기록된 전체 ID의 유일성과 부모 참조를 검증합니다.

In [ ]:
# [B02-03] 전체 추적표를 읽고 명백한 ID·시간 오류를 검사
# [계보표 해석] 파일 열은 cell_id, 추적 시작/끝 프레임, 부모 cell_id입니다.
with ZipFile(ZIP_PATH) as z:
    tracks = pd.read_csv(BytesIO(z.read(track_file)), sep=r"\s+", header=None,
                         names=["cell_id", "start", "end", "parent_id"])
# [자료형] Cell ID와 프레임 번호가 정확한 정수가 되도록 맞춥니다.
for col in tracks.columns:
    tracks[col] = tracks[col].astype(int)
# [기본 무결성] 중복 ID, 음수/0 ID, 역전된 시간 구간을 검출합니다.
assert tracks["cell_id"].is_unique, "중복 cell_id"
assert (tracks["cell_id"] > 0).all()
assert (tracks["start"] <= tracks["end"]).all()
# [계보 무결성] 지정된 부모가 실제 추적표에 존재해야 합니다. 0은 부모 미지정을 의미합니다.
known_ids = set(tracks["cell_id"])
missing_parent_refs = set(tracks.loc[tracks["parent_id"] != 0, "parent_id"]) - known_ids
assert not missing_parent_refs, f"부모 ID가 추적표에 없음: {missing_parent_refs}"
assert not (tracks["cell_id"] == tracks["parent_id"]).any(), "자기 자신을 부모로 참조"
print("Track records:", len(tracks))
print("Root or unknown-parent records:", (tracks["parent_id"]==0).sum())
print("Parent-linked records:", (tracks["parent_id"]!=0).sum())
display(tracks.head(15))
# [교육·재현성] 원본 계보를 해석한 표를 결과 폴더에 저장합니다.
tracks.to_csv(RESULTS_DIR/"track_metadata_69.csv", index=False)

## 4. 첫 프레임 시각화: SEG와 TRA는 다릅니다
영상/SEG/TRA가 같은 픽셀 크기인지 확인하고 나란히 보여 줍니다. 첫 프레임의 TRA 레이블이 추적표의 기간에 나타나는지도 점검합니다.

In [ ]:
# [B02-04] 원본–SEG–TRA 표시 및 1개 프레임의 ID 감사
# [첫 프레임 선택] 세 데이터가 동시에 존재하는 최초 시점을 이용합니다.
FIRST = selected_frames[0]
with ZipFile(ZIP_PATH) as z:
    raw_img = read_tif(z, image_files[FIRST])
    seg0 = read_tif(z, seg_files[FIRST])
    tra0 = read_tif(z, tra_files[FIRST])
# [공간 정합성] RGB 영상이라도 공간 크기 H×W가 SEG·TRA와 같아야 합니다.
gray0 = gray_image(raw_img)
assert gray0.shape == seg0.shape == tra0.shape
assert np.issubdtype(seg0.dtype, np.integer) and np.issubdtype(tra0.dtype, np.integer)
# [추적 ID 검사] TRA 라벨의 0은 배경이며, 계보표의 활성 ID와 비교합니다.
observed = set(map(int, np.unique(tra0))) - {0}
expected = set(tracks.loc[(tracks.start <= FIRST)&(tracks.end >= FIRST), "cell_id"])
print("Frame:", FIRST, "image:", raw_img.shape, "SEG:", seg0.shape, "TRA:", tra0.shape)
print("TRA IDs:", sorted(observed))
print("Missing expected:", sorted(expected-observed), "Unexpected:", sorted(observed-expected))
# 첫 프레임은 일치한다고 사전에 확인되었으므로 재검증
assert observed == expected
# [시각화] SEG의 객체 번호와 TRA의 지속 ID가 별도 라벨임을 비교합니다.
fig, axs = plt.subplots(1, 3, figsize=(10, 6))
axs[0].imshow(raw_img, cmap="gray" if raw_img.ndim==2 else None); axs[0].set_title("Original")
axs[1].imshow(seg0, cmap="nipy_spectral", interpolation="nearest"); axs[1].set_title("SEG instance labels")
axs[2].imshow(tra0, cmap="nipy_spectral", interpolation="nearest"); axs[2].set_title("TRA track IDs")
for a in axs: a.axis("off")
fig.tight_layout()
# [결과 저장] 실습 해설서에서 재사용할 수 있도록 PNG로 출력합니다.
fig.savefig(RESULTS_DIR/"frame0_original_SEG_TRA.png", dpi=160)
plt.show()

## 5. TRA 기반 형태 시계열 구축
`regionprops_table(TRA)`로 **TRA의 cell_id**를 사용해 세포별 면적과 장축·단축을 측정합니다. 영상 강도는 RGB일 때 회색조로 변환하여 기록합니다. *SEG ID로 시간 연결하지 않습니다.*

In [ ]:
# [B02-05] 여러 프레임의 TRA 개체를 측정하고 ID-시간 시계열 생성
# [측정 특성] TRA 개체별 형상과 회색조 밝기를 추출합니다. 픽셀 크기 보정 전 단위입니다.
PROPS = ("label", "area", "major_axis_length", "minor_axis_length", "centroid", "mean_intensity")

# [재사용 함수] 각 프레임의 TRA 라벨을 cell_id로 해석하여 형태 특성을 계산합니다.
def measure_track_frame(img, tra, frame):
    # 원본이 RGB일 수 있으므로 intensity_image용 2D 회색조를 만듭니다.
    gray = gray_image(img)
    if gray.shape != tra.shape or tra.ndim != 2:
        raise ValueError(f"Frame {frame}: 공간 배열 크기가 다릅니다.")
    if not np.issubdtype(tra.dtype, np.integer):
        raise TypeError("TRA 마스크는 정수 레이블이어야 합니다.")
    # regionprops_table의 label 값은 이 셀에서만 TRA의 지속 추적 ID입니다.
    table = pd.DataFrame(regionprops_table(tra, intensity_image=gray, properties=PROPS))
    table = table.rename(columns={
        "label":"cell_id", "area":"area_px2", "major_axis_length":"major_axis_px",
        "minor_axis_length":"minor_axis_px", "mean_intensity":"mean_intensity_gray"
    })
    # 촬영 간격 5분을 적용해 프레임 번호를 경과 시간(분)으로 바꿉니다.
    table.insert(0, "frame", int(frame))
    table.insert(1, "elapsed_minutes", int(frame) * MINUTES_PER_FRAME)
    return table

# [누적 저장] 개별 관측값과 프레임별 추적 ID 정합성 검사를 분리합니다.
measurements=[]
qc=[]
# [프레임 반복] 영상/SEG/TRA의 픽셀 배열 정합성과 계보표 ID를 매 시점 검사합니다.
with ZipFile(ZIP_PATH) as z:
    for t in selected_frames:
        img = read_tif(z, image_files[t]); seg = read_tif(z, seg_files[t]); tra = read_tif(z, tra_files[t])
        if gray_image(img).shape != seg.shape or seg.shape != tra.shape:
            raise ValueError(f"Frame {t}: 원본/SEG/TRA 공간 크기 불일치")
        table = measure_track_frame(img, tra, t)
        # 메타데이터의 start/end를 포함 구간으로 보고 활성 추적 ID를 구합니다.
        expected_t = set(tracks.loc[(tracks.start <= t)&(tracks.end >= t), "cell_id"])
        actual_t = set(table["cell_id"].astype(int))
        # 누락 ID와 예기치 않은 ID를 각각 계산하여 문제 있는 프레임을 찾습니다.
        qc.append({"frame":t,"track_ids_observed":len(actual_t),"expected_in_metadata":len(expected_t),
                   "missing_id_count":len(expected_t-actual_t),"unexpected_id_count":len(actual_t-expected_t)})
        if not table.empty:
            measurements.append(table)
if not measurements:
    raise RuntimeError("분석 기간에 관찰된 TRA 개체가 없습니다.")
# [시계열 생성] 반복 관측을 cell_id와 시간 열이 있는 긴 형태(long-form) 표로 결합합니다.
observations = pd.concat(measurements, ignore_index=True)
frame_qc = pd.DataFrame(qc)
print("TRA observations:", len(observations), "/ frames with cells:", observations.frame.nunique())
print("Frame-level QC:")
display(frame_qc.head())
# [중요] QC가 0인지 전체 행을 확인해야 합니다. 경고를 출력하되 임의로 데이터를 삭제하지 않습니다.
if frame_qc[["missing_id_count","unexpected_id_count"]].to_numpy().sum():
    print("주의: 일부 프레임에서 계보표 기간과 실제 TRA 라벨이 일치하지 않습니다. QC CSV를 확인하세요.")
# [결과 저장] B03에서 추적 시계열을 재사용할 수 있게 CSV 파일로 남깁니다.
observations.to_csv(RESULTS_DIR/"track_observations_first20.csv", index=False)
frame_qc.to_csv(RESULTS_DIR/"frame_track_id_qc_first20.csv", index=False)
display(observations.head(10))

## 6. 세포별 시간 변화와 예시 추적
관측된 TRA ID 중 **여러 프레임에서 관찰된 세포**를 선택하여 면적·장축 변화 그래프를 만듭니다. 추적 중단이나 자손 출현을 성장률로 오인하지 않도록 그래프 범위를 해당 세포의 실제 관측 시점으로 제한합니다.

In [ ]:
# [B02-06] 관측 횟수가 많은 세포 예시를 선택해 면적·길이 곡선 작성
# [예시 추적 선택] 관측 시점이 많은 cell_id를 골라 변화 곡선을 보여 줍니다.
counts = observations.groupby("cell_id").size().sort_values(ascending=False)
EXAMPLE_IDS = counts.head(4).index.astype(int).tolist()
print("Example IDs:", EXAMPLE_IDS)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# [동일 ID 연결] TRA ID가 동일한 관측점만 시간순으로 연결합니다.
for cell_id in EXAMPLE_IDS:
    d = observations.loc[observations.cell_id == cell_id].sort_values("frame")
    axes[0].plot(d.elapsed_minutes, d.area_px2, marker="o", label=f"ID {cell_id}")
    axes[1].plot(d.elapsed_minutes, d.major_axis_px, marker="o", label=f"ID {cell_id}")
# [축 해석] 면적 단위는 px², 장축 길이는 px로 실제 µm가 아닙니다.
axes[0].set(xlabel="Elapsed min (frame 0 = 0)", ylabel="Area (px²)", title="Observed area vs time")
axes[1].set(xlabel="Elapsed min (frame 0 = 0)", ylabel="Major axis (px)", title="Observed major axis vs time")
for a in axes:
    a.legend(fontsize=8); a.grid(alpha=0.25)
fig.tight_layout()
# [결과 저장] 이는 성장률 모델의 적합 결과가 아닌 원시 관측 시계열입니다.
fig.savefig(RESULTS_DIR/"example_tracks_area_length.png", dpi=160)
plt.show()

## 7. 부모–자식 관계 및 관찰 범위
같은 부모에서 갈라지는 추적 ID를 찾고 관계표를 만듭니다. **부모가 지정된 두 자식의 존재는 주석된 계보 관계**를 의미합니다. 실제 분열 시점·세대시간을 자동 확정하는 것은 다음 B03의 주제입니다.

In [ ]:
# [B02-07] 부모 기준으로 자식 그룹을 집계하고 실제 예시를 확인
# [계보 연결] parent_id=0인 행은 부모 미지정이므로 자식 집계에서 제외합니다.
children = tracks.loc[tracks.parent_id != 0].copy()
# [부모별 집계] 한 부모에 연결된 자식 수와 자식 ID 목록을 요약합니다.
family = children.groupby("parent_id").agg(
    child_count=("cell_id","size"),
    child_ids=("cell_id",lambda s: ", ".join(map(str, sorted(s.astype(int)))))
).reset_index().sort_values(["child_count","parent_id"], ascending=[False,True])
print("Parents with annotated children:", len(family))
display(family.head(12))
# [실제 사례] 5번 부모의 자식 관계가 어떻게 기록되어 있는지 확인합니다.
example_parent = 5
print(f"Children of cell {example_parent}:")
display(children.loc[children.parent_id==example_parent, ["cell_id","start","end","parent_id"]])
# [결과 저장] 분열 시간·세대시간의 추정은 여기서 수행하지 않습니다.
family.to_csv(RESULTS_DIR/"parent_child_summary.csv", index=False)

## 8. 결과 해석과 다음 단계

**이 Notebook에서 검증하는 주장**: 동일 TRA ID의 프레임별 형태 측정값을 시간 순서로 연결할 수 있고, 메타데이터의 부모 관계를 구조적으로 확인할 수 있습니다.

**아직 검증하지 않은 주장**: 지수성장 여부, 실제 세포 길이(µm), 단일세포 세대시간, 생균수(CFU), 배양 조건에 따른 성장 차이. 이 결과에는 픽셀 크기 정보가 적용되지 않았습니다.

**연습문제:** ① `MAX_FRAMES=None`으로 전체 193프레임 분석 후 QC 점검 ② Cell 5/8/9 계보의 연결 확인 ③ 같은 TRA ID에서 객체가 소실되는 프레임의 원본 이미지 검사.

**다음 B03:** 분열 사건과 관찰 종료/추적 실패를 구별하고, 적절한 추적에 한해 성장률 및 세대시간의 추정 가능성을 검증합니다.